In [1]:
import os
import torch
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import numpy as np
import pandas as pd
import warnings
from tqdm.auto import tqdm
import timm
import traceback

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

dataset_root = r"c:\Users\hardi\OneDrive\Desktop\MedAIExplainableFractureDetection\balanced_augmented_dataset\test"
classes = sorted(os.listdir(dataset_root))
print("Classes:", classes)

# Helper for standard models
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

def evaluate_model(model_fn, dataset_path, predict_fn):
    y_true = []
    y_pred = []
    for idx, cls in enumerate(classes):
        cls_dir = os.path.join(dataset_path, cls)
        for img_name in tqdm(os.listdir(cls_dir), desc=cls, leave=False):
            img_path = os.path.join(cls_dir, img_name)
            try:
                pred_idx = predict_fn(img_path)
                y_true.append(idx)
                y_pred.append(pred_idx)
            except Exception as e:
                pass
    
    print("\n--- Classification Report ---")
    print(classification_report(y_true, y_pred, target_names=classes, digits=4))
    print("--- Confusion Matrix ---")
    print(confusion_matrix(y_true, y_pred))
    
    # Return macro results
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {'Precision': p, 'Recall': r, 'F1-Score': f1}

results_summary = {}


C:\Users\hardi\AppData\Local\Programs\Python\Python313\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
C:\Users\hardi\AppData\Local\Programs\Python\Python313\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may

Using device: cpu
Classes: ['Comminuted', 'Greenstick', 'Healthy', 'Oblique', 'Oblique_Displaced', 'Spiral', 'Transverse', 'Transverse_Displaced']


In [2]:
# 1. Evaluate MaxViT
import torch.nn as nn

try:
    model_maxvit = timm.create_model('maxvit_rmlp_small_rw_224.sw_in1k', pretrained=False)
    model_maxvit.head = nn.Linear(model_maxvit.head.in_features, 8)
    
    ck = torch.load(r'c:\Users\hardi\OneDrive\Desktop\MedAIExplainableFractureDetection\outputs\cross_validation\best_maxvit.pth', map_location='cpu')
    model_maxvit.load_state_dict(ck.get('model_state_dict', ck), strict=False)
    model_maxvit.to(device)
    model_maxvit.eval()

    def predict_maxvit(img_path):
        img = Image.open(img_path).convert('RGB')
        inp = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model_maxvit(inp)
        return int(out.argmax(dim=1).item())

    print("\nEvaluating MaxViT...")
    results_summary['MaxViT'] = evaluate_model(model_maxvit, dataset_root, predict_maxvit)
except Exception as e:
    print("MaxViT Error:", e)
    traceback.print_exc()



Evaluating MaxViT...


Comminuted:   0%|          | 0/17 [00:00<?, ?it/s]

Greenstick:   0%|          | 0/13 [00:00<?, ?it/s]

Healthy:   0%|          | 0/10 [00:00<?, ?it/s]

Oblique:   0%|          | 0/9 [00:00<?, ?it/s]

...

Spiral:   0%|          | 0/13 [00:00<?, ?it/s]

Transverse:   0%|          | 0/17 [00:00<?, ?it/s]

Transverse_Displaced:   0%|          | 0/17 [00:00<?, ?it/s]


--- Classification Report ---
MaxViT Error: Number of classes, 0, does not match size of target_names, 8. Try specifying the labels parameter


Traceback (most recent call last):
  File "C:\Users\hardi\AppData\Local\Temp\ipykernel_33744\4246539368.py", line 21, in <module>
    results_summary['MaxViT'] = evaluate_model(model_maxvit, dataset_root, predict_maxvit)
                                ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\hardi\AppData\Local\Temp\ipykernel_33744\3239399031.py", line 44, in evaluate_model
    print(classification_report(y_true, y_pred, target_names=classes, digits=4))
          ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\hardi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
  File "C:\Users\hardi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py", line 2945, in classification_report
    raise ValueError(
    ...<3 lines>...
    )
ValueError: Number of classes, 0, does not match siz

In [3]:
# 2. Evaluate hypercolumn_cbam_densenet169_focal
import sys
if r"c:\Users\hardi\OneDrive\Desktop\MedAIExplainableFractureDetection" not in sys.path:
    sys.path.insert(0, r"c:\Users\hardi\OneDrive\Desktop\MedAIExplainableFractureDetection")

try:
    import visualize_xgradcam as vxc
    model_densenet = vxc.get_model('densenet169', num_classes=8, pretrained=False)
    ck2 = torch.load(r'c:\Users\hardi\OneDrive\Desktop\MedAIExplainableFractureDetection\outputs\cross_validation\best_hypercolumn_cbam_densenet169_focal.pth', map_location='cpu')
    model_densenet.load_state_dict(ck2.get('model_state_dict', ck2), strict=False)
    model_densenet.to(device)
    model_densenet.eval()

    def predict_densenet(img_path):
        img = Image.open(img_path).convert('RGB')
        inp = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model_densenet(inp)
        return int(out.argmax(dim=1).item())

    print("\nEvaluating Hypercolumn CBAM DenseNet169 Focal...")
    results_summary['HC_CBAM_DenseNet169_Focal'] = evaluate_model(model_densenet, dataset_root, predict_densenet)
except Exception as e:
    print("Hypercolumn DenseNet Error:", e)
    traceback.print_exc()



Evaluating Hypercolumn CBAM DenseNet169 Focal...


Comminuted:   0%|          | 0/17 [00:00<?, ?it/s]

Greenstick:   0%|          | 0/13 [00:00<?, ?it/s]

Healthy:   0%|          | 0/10 [00:00<?, ?it/s]

Oblique:   0%|          | 0/9 [00:00<?, ?it/s]

Oblique_Displaced:   0%|          | 0/17 [00:00<?, ?it/s]

Spiral:   0%|          | 0/13 [00:00<?, ?it/s]

Transverse:   0%|          | 0/17 [00:00<?, ?it/s]

Transverse_Displaced:   0%|          | 0/17 [00:00<?, ?it/s]


--- Classification Report ---
                      precision    recall  f1-score   support

          Comminuted     0.0000    0.0000    0.0000        17
          Greenstick     0.0000    0.0000    0.0000        13
             Healthy     0.2727    0.6000    0.3750        10
             Oblique     0.0000    0.0000    0.0000         9
   Oblique_Displaced     0.0833    0.0588    0.0690        17
              Spiral     0.2500    0.5000    0.3333        12
          Transverse     0.0417    0.0588    0.0488        17
Transverse_Displaced     0.0000    0.0000    0.0000        17

            accuracy                         0.1250       112
           macro avg     0.0810    0.1522    0.1033       112
        weighted avg     0.0701    0.1250    0.0871       112

--- Confusion Matrix ---
[[0 6 2 5 3 1 0 0]
 [0 0 4 0 0 0 9 0]
 [0 0 6 0 0 1 3 0]
 [0 1 2 0 0 6 0 0]
 [0 3 5 3 1 4 0 1]
 [0 0 0 0 1 6 5 0]
 [0 4 2 3 6 1 1 0]
 [0 1 1 3 1 5 6 0]]


In [4]:
# 3. Evaluate YOLOv26m-cls (using best.pt from finetuning)
try:
    from ultralytics import YOLO

    yolo_path = r'c:\Users\hardi\OneDrive\Desktop\MedAIExplainableFractureDetection\outputs\yolo_cls_finetune\yolo_cls_ft\weights\best.pt'
    if not os.path.exists(yolo_path):
        yolo_path = r'c:\Users\hardi\OneDrive\Desktop\MedAIExplainableFractureDetection\yolov8n-cls.pt' # fallback
    model_yolo = YOLO(yolo_path)

    yolo_names = model_yolo.names
    name_to_idx = {name: idx for idx, name in enumerate(classes)}

    def predict_yolo(img_path):
        res = model_yolo(img_path, verbose=False)
        yolo_cls_name = res[0].names[res[0].probs.top1]
        yolo_cls_name = yolo_cls_name.replace(' ', '_')
        return name_to_idx.get(yolo_cls_name, 0)

    print("\nEvaluating YOLO Model...")
    results_summary['YOLOv26m-cls'] = evaluate_model(model_yolo, dataset_root, predict_yolo)
except Exception as e:
    print("YOLO Error:", e)
    traceback.print_exc()



Evaluating YOLO Model...


Comminuted:   0%|          | 0/17 [00:00<?, ?it/s]

Greenstick:   0%|          | 0/13 [00:00<?, ?it/s]

Healthy:   0%|          | 0/10 [00:00<?, ?it/s]

Oblique:   0%|          | 0/9 [00:00<?, ?it/s]

Oblique_Displaced:   0%|          | 0/17 [00:00<?, ?it/s]

Spiral:   0%|          | 0/13 [00:00<?, ?it/s]

Transverse:   0%|          | 0/17 [00:00<?, ?it/s]

Transverse_Displaced:   0%|          | 0/17 [00:00<?, ?it/s]


--- Classification Report ---
                      precision    recall  f1-score   support

          Comminuted     0.9444    1.0000    0.9714        17
          Greenstick     0.8667    1.0000    0.9286        13
             Healthy     0.7692    1.0000    0.8696        10
             Oblique     0.8000    0.8889    0.8421         9
   Oblique_Displaced     0.8000    0.7059    0.7500        17
              Spiral     1.0000    1.0000    1.0000        12
          Transverse     0.8824    0.8824    0.8824        17
Transverse_Displaced     0.8333    0.5882    0.6897        17

            accuracy                         0.8661       112
           macro avg     0.8620    0.8832    0.8667       112
        weighted avg     0.8659    0.8661    0.8601       112

--- Confusion Matrix ---
[[17  0  0  0  0  0  0  0]
 [ 0 13  0  0  0  0  0  0]
 [ 0  0 10  0  0  0  0  0]
 [ 0  0  1  8  0  0  0  0]
 [ 0  1  1  1 12  0  0  2]
 [ 0  0  0  0  0 12  0  0]
 [ 0  0  0  0  2  0 15  0]
 [ 1  1 

In [5]:
# 4. Evaluate RAD-DINO
try:
    from transformers import AutoImageProcessor, AutoModelForImageClassification

    model_id = "microsoft/rad-dino"
    image_processor = AutoImageProcessor.from_pretrained(model_id)

    model_dino = AutoModelForImageClassification.from_pretrained(model_id, num_labels=8, ignore_mismatched_sizes=True)

    pth_path = r'c:\Users\hardi\OneDrive\Desktop\MedAIExplainableFractureDetection\outputs\dinorad\dinorad_best.pth'
    if os.path.exists(pth_path):
        ck = torch.load(pth_path, map_location='cpu')
        
        # Determine if we need to map keys (e.g. if it has backbone. prefixes)
        state_dict = ck.get('model_state_dict', ck)
        mapped_sd = {}
        for k, v in state_dict.items():
            new_k = k.replace('backbone.', 'dinov2.')
            if new_k.startswith('classifier.'):
                if 'weight' in new_k: new_k = 'classifier.weight'
                elif 'bias' in new_k: new_k = 'classifier.bias'
                else: continue
            mapped_sd[new_k] = v
            
        print('Loading mapped state dict from dinorad_best.pth...')
        model_dino.load_state_dict(mapped_sd, strict=False)

    model_dino.to(device)
    model_dino.eval()

    def predict_dino(img_path):
        img = Image.open(img_path).convert('RGB')
        inputs = image_processor(images=img, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model_dino(**inputs)
        pred_dino_idx = int(outputs.logits.argmax(dim=1).item())
        return pred_dino_idx

    print("\nEvaluating RAD-DINO...")
    results_summary['RAD-DINO'] = evaluate_model(model_dino, dataset_root, predict_dino)
except Exception as e:
    print("RAD-DINO Error:", e)
    traceback.print_exc()


The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Dinov2ForImageClassification LOAD REPORT from: microsoft/rad-dino
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading mapped state dict from dinorad_best.pth...


RAD-DINO Error: Error(s) in loading state_dict for Dinov2ForImageClassification:
	size mismatch for classifier.weight: copying a param with shape torch.Size([8, 768]) from checkpoint, the shape in current model is torch.Size([8, 1536]).


Traceback (most recent call last):
  File "C:\Users\hardi\AppData\Local\Temp\ipykernel_33744\2981395017.py", line 26, in <module>
    model_dino.load_state_dict(mapped_sd, strict=False)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\hardi\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\module.py", line 2635, in load_state_dict
    raise RuntimeError(
    ...<3 lines>...
    )
RuntimeError: Error(s) in loading state_dict for Dinov2ForImageClassification:
	size mismatch for classifier.weight: copying a param with shape torch.Size([8, 768]) from checkpoint, the shape in current model is torch.Size([8, 1536]).


In [6]:
# 5. Summary
try:
    if results_summary:
        summary_df = pd.DataFrame(results_summary).T
        print("\n===========================================")
        print("Overall Macro Metrics for Entire Dataset:")
        print("===========================================")
        print(summary_df.to_string())
    else:
        print("No models evaluated successfully.")
except Exception as e:
    print("Summary Error:", e)



Overall Macro Metrics for Entire Dataset:
                           Precision    Recall  F1-Score
HC_CBAM_DenseNet169_Focal   0.080966  0.152206   0.10326
YOLOv26m-cls                0.862004  0.883170   0.86671
